# Notebook 3: Explicabilité (SHAP, LIME, Contrefactuels)

## Credit Scoring avec IA Explicable (XAI)

### Objectifs
- Implémenter SHAP pour l'explicabilité globale et locale
- Implémenter LIME pour les explications locales
- Générer des explications contrefactuelles
- Comparer SHAP et LIME
- Visualiser les explications

In [ ]:
# Import des bibliothèques
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Import des modules du projet
from src.data_loader import load_data
from src.preprocessing import DataPreprocessor, split_data
from src.models.xgboost_model import XGBoostModel
from src.explainability.shap_explainer import SHAPExplainer
from src.explainability.lime_explainer import LIMEExplainer
from src.explainability.counterfactual import CounterfactualExplainer
from src.config import MODELS_DIR

print("✅ Import des bibliothèques réussi")

## 1. Chargement du Modèle et des Données

In [ ]:
# Charger les données
X, y = load_data()
preprocessor = DataPreprocessor()
X_transformed = preprocessor.fit_transform(X)
X_train, X_val, X_test, y_train, y_val, y_test = split_data(X_transformed, y)

print(f"✅ Données chargées: {X_test.shape}")

In [ ]:
# Charger le modèle XGBoost
model = XGBoostModel()
model_path = MODELS_DIR / "xgboost_model.pkl"

if model_path.exists():
    model.load(str(model_path))
    print(f"✅ Modèle chargé depuis {model_path}")
else:
    model.train(X_train, y_train, X_val, y_val)
    model.save(str(model_path))
    print(f"✅ Modèle entraîné et sauvegardé")

## 2. SHAP (SHapley Additive exPlanations)

In [ ]:
# Initialiser l'explainer SHAP
shap_explainer = SHAPExplainer(model, feature_names=preprocessor.feature_names)
shap_explainer.fit(X_train, explainer_type='tree')

print("✅ Explainer SHAP initialisé")

In [ ]:
# Calculer les valeurs SHAP pour le test set
shap_values = shap_explainer.explain(X_test)

print(f"✅ Valeurs SHAP calculées: {shap_values.shape}")

In [ ]:
# Importance globale des features (SHAP)
shap_importance = shap_explainer.get_feature_importance(importance_type='mean_abs')

print("\n📊 Top 10 features par importance SHAP:")
print(shap_importance.head(10))

In [ ]:
# Visualiser l'importance globale
fig, ax = plt.subplots(figsize=(12, 8))
sns.barplot(data=shap_importance.head(10), x='importance', y='feature', ax=ax)
ax.set_title('Top 10 Features par Importance SHAP')
ax.set_xlabel('Importance SHAP (Mean Absolute)')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
# Explication locale pour une instance spécifique
instance_idx = 0
local_expl = shap_explainer.get_local_explanation(instance_idx, X_test)

print(f"\n🔍 Explication locale SHAP pour l'instance {instance_idx}:")
print(f"   - Base value: {local_expl['base_value']:.4f}")
print(f"   - Prediction: {local_expl['prediction']:.4f}")
print(f"   - Top 5 features:")
for feat in local_expl['features'][:5]:
    print(f"     * {feat['feature']}: {feat['shap_value']:.4f} ({feat['impact']})")

In [ ]:
# Visualiser l'explication locale (Waterfall)
fig = shap_explainer.plot_waterfall(instance_idx, X_test)
plt.show()

In [ ]:
# Visualiser l'explication locale (Force)
fig = shap_explainer.plot_force(instance_idx, X_test)
plt.show()

In [ ]:
# Visualiser la dépendance pour une feature spécifique
fig = shap_explainer.plot_dependence('credit_amount', X_test)
plt.show()

## 3. LIME (Local Interpretable Model-agnostic Explanations)

In [ ]:
# Initialiser l'explainer LIME
lime_explainer = LIMEExplainer(model, feature_names=preprocessor.feature_names)
lime_explainer.fit(X_train)

print("✅ Explainer LIME initialisé")

In [ ]:
# Expliquer une instance avec LIME
instance = X_test.iloc[instance_idx]
lime_explanation = lime_explainer.explain_instance(instance.values)

print(f"\n🔍 Explication LIME pour l'instance {instance_idx}:")
print(f"   - Classe prédite: {lime_explanation.top_labels[0]}")
print(f"   - Probabilité: {lime_explanation.local_pred[1]:.4f}")
print(f"   - Top 5 features:")
for feature, contribution in lime_explanation.as_list()[:5]:
    print(f"     * {feature}: {contribution:.4f}")

In [ ]:
# Explication structurée LIME
lime_local_expl = lime_explainer.get_local_explanation(instance.values, lime_explanation)

print(f"\n📊 Explication structurée LIME:")
print(f"   - Probabilité prédite: {lime_local_expl['predicted_probability']:.4f}")
print(f"   - Intercept: {lime_local_expl['intercept']:.4f}")
print(f"   - Top 5 features:")
for feat in lime_local_expl['features'][:5]:
    print(f"     * {feat['feature']}: {feat['contribution']:.4f} ({feat['impact']})")

In [ ]:
# Visualiser l'explication LIME
fig = lime_explainer.plot_explanation(instance.values, lime_explanation)
plt.show()

## 4. Comparaison SHAP vs LIME

In [ ]:
# Comparer SHAP et LIME pour la même instance
comparison = lime_explainer.compare_with_shap(
    instance.values,
    shap_explainer,
    X_test,
    instance_idx
)

print("\n📊 Comparaison SHAP vs LIME:")
print(f"   - Top 5 SHAP: {[f['feature'] for f in comparison['shap']['features'][:5]]}")
print(f"   - Top 5 LIME: {[f['feature'] for f in comparison['lime']['features'][:5]]}")
print(f"   - Features communes: {comparison['common_top_features']}")
print(f"   - SHAP uniquement: {comparison['shap_only_top_features']}")
print(f"   - LIME uniquement: {comparison['lime_only_top_features']}")

## 5. Explications Contrefactuelles

In [ ]:
# Initialiser l'explainer contrefactuel
cf_explainer = CounterfactualExplainer(model, feature_names=preprocessor.feature_names)

print("✅ Explainer contrefactuel initialisé")

In [ ]:
# Trouver une instance rejetée
rejected_indices = np.where(model.predict(X_test) == 0)[0]

if len(rejected_indices) > 0:
    rejected_idx = rejected_indices[0]
    rejected_instance = X_test.iloc[rejected_idx]
    
    print(f"\n🔍 Instance rejetée {rejected_idx}:")
    print(f"   - Prédiction: {model.predict(rejected_instance.values.reshape(1, -1))[0]}")
    print(f"   - Probabilité: {model.predict_proba(rejected_instance.values.reshape(1, -1))[0]:.4f}")
    
    # Générer une explication contrefactuelle
    cf = cf_explainer.generate_counterfactual(rejected_instance, target_class=1)
    
    if cf['success']:
        print(f"\n✅ Explication contrefactuelle générée:")
        print(f"   - Prédiction originale: {cf['original_prediction']:.4f}")
        print(f"   - Prédiction contrefactuelle: {cf['counterfactual_prediction']:.4f}")
        print(f"   - Distance: {cf['distance']:.4f}")
        print(f"   - Itérations: {cf['iterations']}")
        print(f"   - Top 5 changements:")
        for change in cf['changes'][:5]:
            print(f"     * {change['feature']}: {change['original_value']:.2f} -> {change['counterfactual_value']:.2f} ({change['direction']})")
        
        # Explication textuelle
        print(f"\n📝 Explication textuelle:")
        print(cf_explainer.explain_counterfactual(cf))
    else:
        print(f"\n⚠️ Impossible de générer une explication contrefactuelle")
else:
    print("\n⚠️ Aucune instance rejetée trouvée dans le test set")

In [ ]:
# Générer plusieurs explications contrefactuelles
if len(rejected_indices) > 0:
    rejected_idx = rejected_indices[0]
    rejected_instance = X_test.iloc[rejected_idx]
    
    cf_list = cf_explainer.generate_multiple_counterfactuals(
        rejected_instance,
        target_class=1,
        num_counterfactuals=3
    )
    
    print(f"\n📊 {len(cf_list)} explications contrefactuelles générées:")
    for i, cf in enumerate(cf_list):
        print(f"\n   Explication {i+1}:")
        print(f"     - Prédiction: {cf['counterfactual_prediction']:.4f}")
        print(f"     - Distance: {cf['distance']:.4f}")
        print(f"     - Top 3 changements:")
        for change in cf['changes'][:3]:
            print(f"       * {change['feature']}: {change['original_value']:.2f} -> {change['counterfactual_value']:.2f}")

## 6. Analyse des Explications

In [ ]:
# Comparaison des top features par méthode
print("\n📊 Comparaison des top 5 features par méthode:")

shap_top = [f['feature'] for f in shap_importance.head(5)['feature']]
lime_top = [f['feature'] for f in lime_local_expl['features'][:5]]

comparison_df = pd.DataFrame({
    'SHAP': shap_top,
    'LIME': lime_top
})

print(comparison_df)

In [ ]:
# Analyse de la cohérence des explications
common_features = set(shap_top) & set(lime_top)
shap_only = set(shap_top) - set(lime_top)
lime_only = set(lime_top) - set(shap_top)

print(f"\n📊 Analyse de la cohérence:")
print(f"   - Features communes: {len(common_features)} ({common_features})")
print(f"   - SHAP uniquement: {len(shap_only)} ({shap_only})")
print(f"   - LIME uniquement: {len(lime_only)} ({lime_only})")
print(f"   - Taux de concordance: {len(common_features)/5*100:.1f}%")

## 7. Résumé

In [ ]:
print("\n" + "="*60)
print("🔍 RÉSUMÉ DE L'EXPLICABILITÉ")
print("="*60)

print("\n📊 Méthodes implémentées:")
print("   1. SHAP (SHapley Additive exPlanations)")
print("      - Explicabilité globale et locale")
print("      - Basé sur la théorie des jeux")
print("      - Propriétés mathématiques garanties")

print("   2. LIME (Local Interpretable Model-agnostic Explanations)")
print("      - Explicabilité locale")
print("      - Model-agnostic")
print("      - Approximation locale linéaire")

print("   3. Explications Contrefactuelles")
print("      - Répond à "Que faut-il changer ?"")
print("      - Explications actionnables")
print("      - Optimisation par gradient")

print("\n📊 Insights clés:")
print(f"   - Top feature SHAP: {shap_importance.iloc[0]['feature']}")
print(f"   - Top feature LIME: {lime_local_expl['features'][0]['feature']}")
print(f"   - Taux de concordance SHAP/LIME: {len(common_features)/5*100:.1f}%")

print("\n✅ Explicabilité implémentée avec succès")
print("="*60)